# 01 - Exploratory data analysis (synthetic data)

All data in this notebook is synthetic. The exchange-rate series is an invented random walk, so nothing here describes the real market.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt

from fxhedge.config import SimulatorConfig
from fxhedge.sample_data import simulate_macro
from fxhedge.simulator import simulate_internal_sales

START, END = date(2020, 1, 1), date(2026, 9, 30)
macro = simulate_macro(START, END, seed=42)
sales = simulate_internal_sales(SimulatorConfig(start=START, end=END), START, END, seed=42)
macro.head()

## Seasonality of occupancy

The simulator builds a winter and spring-break peak, a summer bump and a hurricane-season trough.

In [ ]:
monthly = sales.groupby(sales["date"].dt.month)["occupancy_rate"].mean()
axis = monthly.plot(kind="bar", figsize=(8, 3), title="Average occupancy by month (synthetic)")
axis.set_xlabel("Month")
axis.set_ylabel("Occupancy rate")
plt.show()

## Exchange rate

Banking series have gaps on weekends; the feature pipeline forward-fills them.

In [ ]:
fx = macro.set_index("date")["usdmxn_fix"]
print("Share of missing days:", round(float(fx.isna().mean()), 3))
axis = fx.ffill().plot(figsize=(8, 3), title="USD/MXN (synthetic random walk)")
axis.set_ylabel("MXN per USD")
plt.show()

## What to notice

- Cash-in follows sales with a fixed conversion ratio and some noise.
- A random walk has no seasonality, so a seasonal model should not beat the random-walk baseline on the exchange rate, while it can on sales. The next notebook checks exactly that.